In [ ]:
```xml
<VSCode.Cell language="markdown">
# ⚡ Real-Time Communication Systems Guide

**Building low-latency, bi-directional communication for interactive experiences**

This guide covers:
- WebSocket fundamentals and advanced patterns
- SignalR for .NET real-time applications
- Socket.io for Node.js
- Server-Sent Events (SSE) vs WebSockets
- Real-time architecture patterns
- Scaling real-time systems
- Broadcasting and group messaging
- Connection management and heartbeats
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. WebSocket Architecture

### Server Implementation (Python)
```python
import asyncio
import websockets
import json
from typing import Set
from dataclasses import dataclass

@dataclass
class Client:
    """Represents connected client"""
    id: str
    websocket: websockets.WebSocketServerProtocol
    room: str = None

class RealTimeServer:
    def __init__(self):
        self.clients: Set[Client] = set()
        self.rooms: dict = {}  # room_id -> Set[Client]
    
    async def register(self, websocket, client_id: str):
        """Register new connection"""
        client = Client(id=client_id, websocket=websocket)
        self.clients.add(client)
        
        logger.info(f"Client {client_id} connected. Total: {len(self.clients)}")
        
        try:
            async for message in websocket:
                await self.handle_message(client, message)
        except websockets.exceptions.ConnectionClosed:
            logger.info(f"Client {client_id} disconnected")
        finally:
            self.clients.discard(client)
            if client.room:
                self.rooms[client.room].discard(client)
    
    async def handle_message(self, client: Client, message: str):
        """Process incoming message"""
        data = json.loads(message)
        message_type = data.get('type')
        
        if message_type == 'join_room':
            await self.join_room(client, data['room_id'])
        
        elif message_type == 'send_message':
            await self.broadcast_to_room(
                client.room,
                {
                    'type': 'message',
                    'sender': client.id,
                    'content': data['content'],
                    'timestamp': datetime.now().isoformat()
                },
                exclude=None  # Send to all including sender
            )
        
        elif message_type == 'ping':
            await client.websocket.send(json.dumps({'type': 'pong'}))
    
    async def join_room(self, client: Client, room_id: str):
        """Add client to room"""
        if room_id not in self.rooms:
            self.rooms[room_id] = set()
        
        self.rooms[room_id].add(client)
        client.room = room_id
        
        # Notify room of new member
        await self.broadcast_to_room(room_id, {
            'type': 'user_joined',
            'user_id': client.id,
            'room_id': room_id
        })
    
    async def broadcast_to_room(self, room_id: str, message: dict, exclude=None):
        """Send message to all clients in room"""
        if room_id not in self.rooms:
            return
        
        message_json = json.dumps(message)
        
        for client in self.rooms[room_id]:
            if exclude and client.id == exclude:
                continue
            
            try:
                await client.websocket.send(message_json)
            except websockets.exceptions.ConnectionClosed:
                self.rooms[room_id].discard(client)

# Start server
async def main():
    async with websockets.serve(
        lambda ws, path: RealTimeServer().register(ws, path),
        "0.0.0.0",
        8000
    ):
        await asyncio.Future()  # run forever

asyncio.run(main())
```

### Client Implementation (JavaScript)
```javascript
class RealtimeClient {
  constructor(serverUrl, userId) {
    this.serverUrl = serverUrl;
    this.userId = userId;
    this.ws = null;
    this.reconnectAttempts = 0;
    this.maxReconnectAttempts = 5;
    this.reconnectDelay = 1000;
    this.listeners = new Map();
  }
  
  connect() {
    return new Promise((resolve, reject) => {
      try {
        this.ws = new WebSocket(this.serverUrl);
        
        this.ws.onopen = () => {
          console.log('Connected to server');
          this.reconnectAttempts = 0;
          this.heartbeat();  // Start heartbeat
          resolve();
        };
        
        this.ws.onmessage = (event) => {
          this.handleMessage(JSON.parse(event.data));
        };
        
        this.ws.onerror = (error) => {
          console.error('WebSocket error:', error);
          reject(error);
        };
        
        this.ws.onclose = () => {
          this.attemptReconnect();
        };
      } catch (error) {
        reject(error);
      }
    });
  }
  
  handleMessage(message) {
    const type = message.type;
    
    // Emit to all listeners of this type
    if (this.listeners.has(type)) {
      this.listeners.get(type).forEach(callback => {
        callback(message);
      });
    }
  }
  
  on(type, callback) {
    if (!this.listeners.has(type)) {
      this.listeners.set(type, []);
    }
    this.listeners.get(type).push(callback);
  }
  
  send(type, data) {
    if (this.ws && this.ws.readyState === WebSocket.OPEN) {
      this.ws.send(JSON.stringify({
        type,
        ...data
      }));
    }
  }
  
  heartbeat() {
    setInterval(() => {
      if (this.ws && this.ws.readyState === WebSocket.OPEN) {
        this.send('ping');
      }
    }, 30000);  // Every 30 seconds
  }
  
  attemptReconnect() {
    if (this.reconnectAttempts < this.maxReconnectAttempts) {
      setTimeout(() => {
        this.reconnectAttempts++;
        console.log(`Reconnecting... (${this.reconnectAttempts}/${this.maxReconnectAttempts})`);
        this.connect();
      }, this.reconnectDelay * Math.pow(2, this.reconnectAttempts - 1));
    }
  }
  
  disconnect() {
    if (this.ws) {
      this.ws.close();
    }
  }
}

// Usage
const client = new RealtimeClient('ws://localhost:8000', 'user123');

await client.connect();

client.on('message', (msg) => {
  console.log(`${msg.sender}: ${msg.content}`);
});

client.send('join_room', { room_id: 'chat-room-1' });

client.send('send_message', { content: 'Hello everyone!' });
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Server-Sent Events (SSE) vs WebSockets

### SSE (One-way, simpler)
```typescript
// Server
app.get('/api/events', (req, res) => {
  res.setHeader('Content-Type', 'text/event-stream');
  res.setHeader('Cache-Control', 'no-cache');
  res.setHeader('Connection', 'keep-alive');
  
  // Send initial connection
  res.write('data: {"status": "connected"}\n\n');
  
  // Send updates
  const interval = setInterval(() => {
    const data = {
      timestamp: new Date().toISOString(),
      count: Math.random() * 100
    };
    res.write(`data: ${JSON.stringify(data)}\n\n`);
  }, 1000);
  
  req.on('close', () => {
    clearInterval(interval);
    res.end();
  });
});

// Client
const eventSource = new EventSource('/api/events');

eventSource.onmessage = (event) => {
  const data = JSON.parse(event.data);
  console.log('Received:', data);
};

eventSource.onerror = () => {
  console.error('Connection error');
  eventSource.close();
};
```

### When to Use

| Feature | WebSocket | SSE |
|---------|-----------|-----|
| **Direction** | Bidirectional | Server → Client |
| **Use Case** | Chat, multiplayer | Notifications, live feeds |
| **Complexity** | Higher | Lower |
| **Performance** | Lower latency | Simpler protocol |
| **Browser Support** | All modern | All modern |
| **Automatic Reconnect** | Manual | Built-in |
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Scaling Real-Time Systems

### Horizontal Scaling with Redis
```python
# When you have multiple servers, broadcast via Redis
import redis
from typing import Callable

class RedisRealTimeServer(RealTimeServer):
    def __init__(self):
        super().__init__()
        self.redis = redis.Redis(host='redis', port=6379)
        self.pubsub = self.redis.pubsub()
    
    async def broadcast_to_room(self, room_id: str, message: dict, exclude=None):
        """Broadcast locally AND via Redis"""
        # Local broadcast
        await super().broadcast_to_room(room_id, message, exclude)
        
        # Broadcast via Redis (for other servers)
        channel = f'room:{room_id}'
        self.redis.publish(channel, json.dumps(message))
    
    async def subscribe_to_redis_updates(self, room_id: str):
        """Listen for messages from other servers"""
        channel = f'room:{room_id}'
        self.pubsub.subscribe(channel)
        
        for message in self.pubsub.listen():
            if message['type'] == 'message':
                data = json.loads(message['data'])
                
                # Broadcast to local clients
                await self.broadcast_to_room(
                    room_id,
                    data,
                    exclude=None  # Don't exclude, Redis already does
                )
```

### Load Balancer Configuration
```nginx
# nginx.conf - Sticky sessions for WebSockets
upstream realtime_servers {
    # Sticky sessions: route same client to same server
    ip_hash;
    
    server realtime-server-1:8000 weight=1;
    server realtime-server-2:8000 weight=1;
    server realtime-server-3:8000 weight=1;
}

server {
    listen 80;
    
    location /ws {
        proxy_pass http://realtime_servers;
        
        # WebSocket headers
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection "upgrade";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        
        # Timeouts
        proxy_read_timeout 86400;
        proxy_send_timeout 86400;
    }
}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Real-Time Best Practices

- **Heartbeat**: Send periodic pings to detect dead connections (every 30s)
- **Exponential Backoff**: Retry with increasing delays on disconnect
- **Message Queuing**: Use Redis/RabbitMQ for multi-server broadcasts
- **Connection Limits**: Set max connections per user to prevent abuse
- **Data Compression**: Compress large payloads with gzip/brotli
- **Error Handling**: Graceful degradation when WebSocket unavailable
- **Monitoring**: Track connection count, message latency, error rates
- **Rate Limiting**: Prevent spam (e.g., max 100 messages/minute per user)
</VSCode.Cell>
```